# تقطیر ترکیبی: دادهٔ متنی ChatGPT + logits مدل Qwen

این نوت‌بوک آرشیو JSONL پرسش/پاسخ‌های نوشته‌شده با ChatGPT را روی Google Drive نگه می‌دارد و با اجرای مدل معلم Qwen، logits توکن بعدی را در موقعیت‌های متناظر با پاسخ‌ها استخراج می‌کند. این کار به logits داخلی ChatGPT دسترسی ندارد و ادعا نمی‌کند که خروجی عددی آن از ChatGPT آمده است.

**حالت ذخیره‌سازی پیش‌فرض:** top-k sparse logits (۶۴ مقدار برتر در هر موقعیت پاسخ) به‌جای ماتریس کامل واژگان. این تقریب کم‌حجم‌تری برای KD است؛ logits کامل دقیق‌تر اما بسیار حجیم‌اند و در این اسکریپت پیاده‌سازی نشده‌اند. برای تقطیر توکنی، مدل دانش‌آموز باید tokenizer و واژگان سازگار با Qwen داشته باشد.

آرشیو ورودی را با نمونه‌هایی در قالب `sample_id`, `prompt`, `chatgpt_response` آماده کن. هرگز دادهٔ eval را به train اضافه نکن.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, json
WORK = Path('/content/drive/MyDrive/qwen-moe-chatgpt-logits')
WORK.mkdir(parents=True, exist_ok=True)
INPUT_JSONL = WORK / 'chatgpt_prompt_response_archive.jsonl'
OUTPUT_DIR = WORK / 'qwen_logits_output'
MODEL_ID = 'Qwen/Qwen3.5-2B'  # قبل از اجرا شناسهٔ دقیق مدل و دسترسی آن را تأیید کن
print('Archive:', INPUT_JSONL)
print('Output:', OUTPUT_DIR)
if not INPUT_JSONL.exists():
    raise FileNotFoundError(f'ابتدا فایل JSONL را در Drive قرار بده: {INPUT_JSONL}')


## قالب فایل ورودی

هر خط یک JSON object مستقل است. شناسهٔ `sample_id` باید یکتا و پایدار باشد. نمونهٔ ساختاری زیر را می‌توانی برای بررسی قالب استفاده کنی؛ برای آموزش واقعی، پاسخ‌های باکیفیت و بازبینی‌شدهٔ خودت را جایگزین کن.

In [ ]:
example = {
  'sample_id': 'fa-reasoning-000001',
  'prompt': 'چرا تقسیم بر صفر در حساب معمول تعریف نشده است؟',
  'chatgpt_response': 'در تقسیم a بر b، عددی می‌خواهیم که ضرب آن در b برابر a شود. وقتی b صفر باشد، برای a غیرصفر چنین عددی وجود ندارد؛ و برای a برابر صفر نیز جواب یکتا نیست.'
}
print(json.dumps(example, ensure_ascii=False, indent=2))
print('این فقط نمونهٔ قالب است و خودکار به آرشیو اضافه نمی‌شود.')


In [ ]:
REPO = Path('/content/Qwen-MoE-0.4B-')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/samanramezani1377-hub/Qwen-MoE-0.4B-.git',str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers', 'accelerate', 'safetensors'], check=True)
print('Setup complete. GPU:', subprocess.run([sys.executable, '-c', "import torch; print(torch.cuda.is_available())"], capture_output=True, text=True).stdout.strip())


## مرحلهٔ اول — smoke test کوچک

ابتدا فقط دو نمونه را اجرا کن. این مرحله مدل را دانلود می‌کند و قالب پیام، tokenizer، حافظه و ذخیره‌سازی را آزمایش می‌کند. اگر شناسهٔ مدل اشتباه/غیرقابل‌دسترسی باشد یا GPU حافظهٔ کافی نداشته باشد، اینجا متوقف شو و خطا را بررسی کن.

In [ ]:
subprocess.run([
    sys.executable, str(REPO/'scripts/capture_qwen_logits_for_chatgpt_data.py'),
    '--input-jsonl', str(INPUT_JSONL), '--output-dir', str(OUTPUT_DIR/'smoke_test'),
    '--model', MODEL_ID, '--top-k', '64', '--max-seq-length', '1024',
    '--max-samples', '2', '--device', 'auto'
], check=True)


## مرحلهٔ دوم — استخراج کامل

پس از موفقیت smoke test، استخراج همهٔ نمونه‌ها را اجرا کن. فایل‌های هر نمونه جدا ذخیره می‌شوند؛ اجرای مجدد فایل‌های موجود را رد می‌کند تا در قطع Colab کار از ابتدا تکرار نشود. برای جلوگیری از مصرف بیش‌ازحد دیسک، ابتدا با چند ده یا چند صد نمونه آزمایش کن.

In [ ]:
subprocess.run([
    sys.executable, str(REPO/'scripts/capture_qwen_logits_for_chatgpt_data.py'),
    '--input-jsonl', str(INPUT_JSONL), '--output-dir', str(OUTPUT_DIR/'full'),
    '--model', MODEL_ID, '--top-k', '64', '--max-seq-length', '2048', '--device', 'auto'
], check=True)
print('Index:', OUTPUT_DIR/'full'/'qwen_logits_index.jsonl')
print('Manifest:', OUTPUT_DIR/'full'/'qwen_logits_manifest.json')


## محدودیت‌ها و نکات علمی

- این خروجی **logits معلم Qwen** است که در متن پاسخ ChatGPT و با پیشوند پرامپت محاسبه شده؛ logits ChatGPT نیست.
- خروجی پیش‌فرض فقط top-k logits را نگه می‌دارد، بنابراین توزیع کامل واژگان نیست و برای KD یک تقریب sparse است.
- این اسکریپت استخراج logits را انجام می‌دهد، اما loss آموزش دانش‌آموز یا مدل MoE را به‌تنهایی اجرا نمی‌کند؛ مصرف‌کنندهٔ KD باید هم‌ترازی توکن‌ها و ماسک پاسخ را رعایت کند.
- اگر دانش‌آموز tokenizer متفاوتی دارد، اندیس توکن‌ها مستقیماً قابل انتقال نیست.
- آرشیو ChatGPT را حفظ کن و train/eval را با شناسه‌های مستقل جدا نگه دار. این نوت‌بوک هنوز روی Colab اجرا و تأیید نشده است.